In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../cefr_leveled_texts.csv')

In [3]:
df.head()

,text,label
0,Hi!\nI've been meaning to write for ages and f...,B2
1,﻿It was not so much how hard people found the ...,B2
2,Keith recently came back from a trip to Chicag...,B2
3,"The Griffith Observatory is a planetarium, and...",B2
4,-LRB- The Hollywood Reporter -RRB- It's offici...,B2


In [9]:
label_encoder = LabelEncoder()
df['level_encoded'] = label_encoder.fit_transform(df['label'])
y = df['level_encoded']

In [10]:
df.head()

,text,label,level_encoded
0,Hi!\nI've been meaning to write for ages and f...,B2,3
1,﻿It was not so much how hard people found the ...,B2,3
2,Keith recently came back from a trip to Chicag...,B2,3
3,"The Griffith Observatory is a planetarium, and...",B2,3
4,-LRB- The Hollywood Reporter -RRB- It's offici...,B2,3


In [7]:
label_counts = df.groupby("level_encoded").size().reset_index(name="count")
print(label_counts)

   level_encoded  count
0              0    288
1              1    272
2              2    205
3              3    286
4              4    241
5              5    202


In [25]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
x = vectorizer.fit_transform(df['text'])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [17]:
model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1
)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, n_jobs=-1)

In [18]:
y_pred = model.predict(X_test)

In [19]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.568561872909699


In [20]:
print("Precision:", precision_score(y_test, y_pred, average="weighted"))
print("Recall:", recall_score(y_test, y_pred, average="weighted"))
print("F1:", f1_score(y_test, y_pred, average="weighted"))

Precision: 0.6077465888143478
Recall: 0.568561872909699
F1: 0.5475346827304334


In [24]:
conf_matrix = confusion_matrix(y_test, y_pred)

labels = ["A1", "A2", "B1", "B2", "C1", "C2"]

df_cm = pd.DataFrame(conf_matrix, index=labels, columns=labels)
print(df_cm)


    A1  A2  B1  B2  C1  C2
A1  58   8   0   1   0   0
A2  21  28   0   3   0   0
B1   4   7   4  20   1   0
B2   3   2   1  37  11   0
C1   0   1   1  21  25   3
C2   0   0   0   7  14  18


Common confusions:
        A2 → A1 (21)
        B1 → B2 (20)
        C1 → B2 (21)
        C2 → C1 (14)

In [27]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_

top_features = sorted(
    zip(coefficients[0], feature_names),
    reverse=True
)[:20]

print("Top positive features:")
for coef, feature in top_features:
    print(feature, coef)

Top positive features:
she 1.5388363536681675
you 1.2341128611483452
okay 1.1536498270965907
yes 1.0287943414802567
mom 0.8098773572310884
he 0.6996866386625269
go 0.6578976785971088
can 0.6313180150731731
do you 0.514627559764704
flu 0.49029674341630525
trash 0.46334666197143665
you can 0.44799738560067526
looked 0.44430158517220086
yes it 0.44417564342199223
his car 0.43931306575857465
teacher 0.4319232163892488
car 0.427496169442344
are you 0.415584660701084
let 0.4119911583515361
ll 0.40827841676955673


Strongest features are mostly pronouns, common spoken words, short conversational phrases. Suggests that the model is learning
conversational style, not language proficiency. That could explain confusion A2 <-> A1, B1 <-> B2, C1 confused with B2
('sounds like a conversation so lower level'). CEFR levels criteria: sentence complexity, grammar structures, lexical richness.

In [30]:
# reduce pronoun dominance and increase ngram range to capture grammar
custom_stopwords = ["she", "he", "you", "we", "they", "i", "me", "him", "her"]

vectorizer_v2 = TfidfVectorizer(
    stop_words=custom_stopwords,
    ngram_range=(1, 3),
    max_features=5000
)

# use class weighting
model_v2 = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

In [31]:
x2 = vectorizer_v2.fit_transform(df['text'])

In [32]:
X_train, X_test, y_train, y_test = train_test_split(x2, y, test_size=0.2, random_state=42)

In [33]:
model_v2.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [34]:
y2_pred = model_v2.predict(X_test)

In [35]:
accuracy = accuracy_score(y_test, y2_pred)
print("Accuracy:", accuracy)

Accuracy: 0.6053511705685619


In [36]:
conf_matrix = confusion_matrix(y_test, y2_pred)

labels = ["A1", "A2", "B1", "B2", "C1", "C2"]

df_cm2 = pd.DataFrame(conf_matrix, index=labels, columns=labels)
print(df_cm2)

    A1  A2  B1  B2  C1  C2
A1  60   6   0   1   0   0
A2  18  28   6   0   0   0
B1   3   6  14  12   1   0
B2   0   5   6  24  17   2
C1   0   0   2  11  28  10
C2   0   0   0   2  10  27
